In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [5]:
dataset=pd.read_csv("Social_Network_Ads.csv")

In [6]:
dataset.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [7]:
dataset=pd.get_dummies(dataset,drop_first=True)

In [8]:
dataset=dataset.astype(int)

In [9]:
dataset.head()

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1


In [10]:
dataset=dataset.drop("User ID",axis=1)

In [11]:
dataset["Purchased"].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [12]:
indep=dataset[["Age","EstimatedSalary","Gender_Male"]]
dep=dataset["Purchased"]

In [13]:
indep.head()

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1


In [14]:
dep.head()

0    0
1    0
2    0
3    0
4    0
Name: Purchased, dtype: int32

In [15]:
#split into training set and test
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

In [16]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Create an untuned KNN model
knn = KNeighborsClassifier()

# Define the parameter grid
param_grid = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree']
}

# Create a GridSearchCV object
grid= GridSearchCV(knn, param_grid, cv=5)

# Fit the grid search object to the data
grid.fit(x_train, y_train)


GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'algorithm': ['auto', 'ball_tree', 'kd_tree'],
                         'n_neighbors': [3, 5, 7],
                         'weights': ['uniform', 'distance']})

In [18]:
# Print the best parameters

print(grid.best_params_)

{'algorithm': 'auto', 'n_neighbors': 5, 'weights': 'uniform'}


In [28]:
re=grid.cv_results_

In [29]:
grid_predictions=grid.predict(x_test)

In [30]:
#y_pred = classifier.predict(x_test)
        

In [31]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
        

In [32]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred)

In [33]:
print(clf_report)

              precision    recall  f1-score   support

           0       0.95      0.92      0.93        85
           1       0.87      0.92      0.89        49

    accuracy                           0.92       134
   macro avg       0.91      0.92      0.91       134
weighted avg       0.92      0.92      0.92       134



In [34]:
print(cm)

[[78  7]
 [ 4 45]]


In [35]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)


The f1_macro value for best parameter {'algorithm': 'auto', 'n_neighbors': 5, 'weights': 'uniform'}: 0.9183922682195829


In [36]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])


0.9411764705882354

In [37]:
table=pd.DataFrame.from_dict(re)

In [38]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_algorithm,param_n_neighbors,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002778,0.003983,0.010886,0.006077,auto,3,uniform,"{'algorithm': 'auto', 'n_neighbors': 3, 'weigh...",0.870370,0.905660,0.867925,0.962264,0.924528,0.906150,0.035294,4
1,0.006266,0.007675,0.000000,0.000000,auto,3,distance,"{'algorithm': 'auto', 'n_neighbors': 3, 'weigh...",0.851852,0.905660,0.811321,0.905660,0.924528,0.879804,0.041973,16
2,0.003123,0.006247,0.009372,0.007652,auto,5,uniform,"{'algorithm': 'auto', 'n_neighbors': 5, 'weigh...",0.870370,0.905660,0.867925,0.943396,0.981132,0.913697,0.043512,1
3,0.003124,0.006248,0.000000,0.000000,auto,5,distance,"{'algorithm': 'auto', 'n_neighbors': 5, 'weigh...",0.851852,0.886792,0.849057,0.943396,0.962264,0.898672,0.046560,13
4,0.003333,0.006157,0.009388,0.007665,auto,7,uniform,"{'algorithm': 'auto', 'n_neighbors': 7, 'weigh...",0.870370,0.886792,0.849057,0.943396,0.962264,0.902376,0.043311,6
5,0.003124,0.006249,0.003124,0.006248,auto,7,distance,"{'algorithm': 'auto', 'n_neighbors': 7, 'weigh...",0.851852,0.905660,0.849057,0.943396,0.943396,0.898672,0.041721,10
6,0.004459,0.006154,0.006889,0.007261,ball_tree,3,uniform,"{'algorithm': 'ball_tree', 'n_neighbors': 3, '...",0.870370,0.905660,0.849057,0.962264,0.924528,0.902376,0.039888,6
7,0.000000,0.000000,0.006248,0.007653,ball_tree,3,distance,"{'algorithm': 'ball_tree', 'n_neighbors': 3, '...",0.851852,0.905660,0.792453,0.905660,0.924528,0.876031,0.048327,18
8,0.003720,0.006060,0.009030,0.005279,ball_tree,5,uniform,"{'algorithm': 'ball_tree', 'n_neighbors': 5, '...",0.870370,0.905660,0.867925,0.943396,0.981132,0.913697,0.043512,1
9,0.000660,0.000907,0.001446,0.001412,ball_tree,5,distance,"{'algorithm': 'ball_tree', 'n_neighbors': 5, '...",0.851852,0.886792,0.849057,0.943396,0.962264,0.898672,0.046560,13


In [39]:
#Future prediction

In [40]:
age=float(input("Age:"))
Estimated_salary=float(input("Estimated_salary:"))
sex_male=int(input("Sex Male 0 or 1:"))


Age:30
Estimated_salary:800000
Sex Male 0 or 1:0


In [41]:
Future_Prediction=grid.predict([[age,Estimated_salary,sex_male]])
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[1]


In [42]:
import pickle

In [43]:
filename= "finalized_model_Grid_KNN_classification.sav"

In [44]:
pickle.dump(grid,open(filename,'wb'))

In [45]:
loaded_model = pickle.load(open("finalized_model_Grid_KNN_classification.sav",'rb'))

In [46]:
result= loaded_model.predict([[age,Estimated_salary,sex_male]])

In [47]:
result

array([1])